# ALACC Truck Special Generators comparison

This notebook compares the trucks special generation trips with those of the SW model 

In [50]:
import pandas as pd 
import numpy as np 

import geopandas as gpd
import openmatrix as omx

import warnings
from pandas.errors import PerformanceWarning

warnings.filterwarnings(
    "ignore",
    category=PerformanceWarning,
)

# Bi-county Special Generation Data

In [9]:
def add_total_row(df: pd.DataFrame, label: str = "Total") -> pd.DataFrame:
    """Return a copy of df with a totals row appended."""

    result = df.copy()

    total_row = {}

    for col in result.columns:
        if pd.api.types.is_numeric_dtype(result[col]):
            total_row[col] = result[col].sum(skipna=True)
        else:
            total_row[col] = ""

    total_df = pd.DataFrame([total_row], index=[label])

    return pd.concat([result, total_df])

In [12]:
# In Alacc scenario path: nonres/Inputs/Calib/PORT_SG_2015.DBF"
# File was converted to CSV using CUBE and save to BOX 
path = "../data/external/ccta/PORT_SG_2015.csv"
df = pd.read_csv(path, header = None).dropna()
df.columns = ["taz", "SMALL", "MEDIUM", "COMBO"]
df["taz"] = df["taz"].astype(float).astype(int)
df = df.set_index("taz")
df["total"] = df.sum(axis = 1)
BCM = add_total_row(df[df["total"] > 0])
print("Table 1. Bi-county Model \nPort of Oakland Special Generator")
BCM.style.format("{:,.0f}")

Table 1. Bi-county Model 
Port of Oakland Special Generator


,SMALL,MEDIUM,COMBO,total
2833,74,194,953,"1,221"
2957,126,324,"1,585","2,035"
2967,50,124,612,786
3163,100,254,"1,245","1,599"
3165,70,186,901,"1,157"
3166,40,104,508,652
3167,16,40,198,254
Total,476,"1,226","6,002","7,704"


Note: All these TAZs are around the Port of Oakland.

# State Wide Model 

In [13]:
# SW Trip Generation projected to TM-1.6 Zoning system
path = "../data/processed/truck_trip_generation_zone.csv"
sw_generation = pd.read_csv(path)
sw_generation =sw_generation.set_index("TAZ1454")

In [14]:
# These TAZs are comparable with the the area covered by the SPECIAL GENERATORS in the Bi-county Model
# This inspection was done manually. 
port_of_oakland_tazs = [988,965, 966]

sw_generation["small"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^LT.*production$")]].sum(axis = 1)
sw_generation["medium"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^MT.*production$")]].sum(axis = 1)
sw_generation["large"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^HT.*production$")]].sum(axis = 1)
sw_generation["total"] = sw_generation[["small", "medium", "large"]].sum(axis =1)

production_port_of_oakland = sw_generation[sw_generation.index.isin(port_of_oakland_tazs)][["small", "medium", "large"]]

In [15]:
port_of_oakland_tazs = [988,965, 966]
production_port_of_oakland = sw_generation[sw_generation.index.isin(port_of_oakland_tazs)][["small", "medium", "large"]]

In [16]:
#Statewide Transportation Logistic Nodes (TLN) projected to TM-1.6 Zoning System
path = "../data/processed/truck_trip_generation_tln.csv"
tln = pd.read_csv(path)

tln["small"] = tln[tln.columns[tln.columns.str.contains(r"^LT.*production$")]].sum(axis = 1)
tln["medium"] = tln[tln.columns[tln.columns.str.contains(r"^MT.*production$")]].sum(axis = 1)
tln["large"] = tln[tln.columns[tln.columns.str.contains(r"^HT.*production$")]].sum(axis = 1)
tln["total"] = tln[["small", "medium", "large"]].sum(axis =1)

In [17]:
tln[["TAZ1454", "small", "medium", "large"]]

,TAZ1454,small,medium,large
0,142.0,0.0,42.229202,129.498908
1,313.0,0.0,14.307400,56.660299
2,965.0,0.0,510.502294,1015.344990
3,1062.0,0.0,135.852806,344.753600
4,239.0,0.0,66.932599,173.411304
5,874.0,0.0,4.811900,29.225001


In [18]:
sw_port_of_oakland = production_port_of_oakland.merge(
    tln[["TAZ1454", "small", "medium", "large"]], 
    left_index = True, 
    right_on = "TAZ1454",
    how = 'left', 
    suffixes=("_taz", "_tln")
).set_index("TAZ1454").fillna(0)

sw_port_of_oakland["total"] = sw_port_of_oakland.sum(axis =1)
sw_port_of_oakland.index = sw_port_of_oakland.index.astype(int)
sw_port_of_oakland = add_total_row(sw_port_of_oakland)
print("Table 2. SW Model \nPort of Oakland TAZ and TLN demand")
sw_port_of_oakland.style.format("{:,.0f}")

Table 2. SW Model 
Port of Oakland TAZ and TLN demand


,small_taz,medium_taz,large_taz,small_tln,medium_tln,large_tln,total
965,42,24,18,0,511,"1,015","1,609"
966,18,10,8,0,0,0,35
988,182,50,15,0,0,0,246
Total,241,83,41,0,511,"1,015","1,891"


In [24]:
sw_port_of_oakland.columns

Index(['small_taz', 'medium_taz', 'large_taz', 'small_tln', 'medium_tln',
       'large_tln', 'total'],
      dtype='str')

In [51]:
all_tln = sw_generation.merge(tln, on = 'TAZ1454', how = "inner",  suffixes=("_taz", "_tln"))
all_tln = all_tln.set_index(["TAZ1454",'zone_name'])
all_tln = all_tln[['small_taz', 'medium_taz', 'large_taz', 'small_tln', 'medium_tln',
       'large_tln']]
all_tln['total'] = all_tln.sum(axis = 1)
all_tln = add_total_row(all_tln)
print("Table 3. SW Model \nTAZ and TLN demand at TAZ that have a TLN")
all_tln.style.format("{:,.0f}")

Table 3. SW Model 
TAZ and TLN demand at TAZ that have a TLN


,small_taz,medium_taz,large_taz,small_tln,medium_tln,large_tln,total
"(142, 'PORT OF SAN FRANCISCO')",277,571,503,0,42,129,"1,523"
"(239, 'SFO AIRPORT')",44,27,29,0,67,173,341
"(313, 'PORT OF REDWOOD CITY')",226,120,72,0,14,57,489
"(874, 'OAK AIRPORT')",29,16,13,0,5,29,92
"(965, 'PORT OF OAKLAND')",42,24,18,0,511,"1,015","1,609"
"(1062, 'PORT OF RICHMOND')",233,105,59,0,136,345,877
Total,852,862,694,0,775,"1,749","4,931"
